# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to explore the FAIR² dataset using the `mlcroissant` library and the Croissant schema.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`


In [ ]:
# Install the mlcroissant library if needed
!pip install mlcroissant

## 1. Data Loading
Load dataset metadata and preview its description using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant JSON-LD schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load metadata and dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset title: {metadata.name}\nDescription: {metadata.description}")

## 2. Data Overview
List all available record sets, their fields, and corresponding `@id`s. This helps us select which data to load for analysis.

**Note:** All references to dataset entities are made by their `@id` values as per the Croissant schema.

In [ ]:
# List all record sets and their fields by @id
print('Available Record Sets:')
for record_set in metadata.record_sets:
    print(f"- RecordSet @id: {record_set.id}, name: {record_set.name if hasattr(record_set, 'name') else ''}")
    if hasattr(record_set, 'fields') and record_set.fields:
        print('  Fields:')
        for field in record_set.fields:
            print(f"    - Field @id: {field.id}, name: {field.name if hasattr(field, 'name') else ''}, dataType: {getattr(field, 'data_type', '')}")
    if hasattr(record_set, 'columns') and record_set.columns:
        print('  Columns:')
        for col in record_set.columns:
            print(f"    - Column @id: {col.id}, name: {col.name}")

## 3. Data Extraction
Select record sets by their `@id` (from the previous overview) and load full records into dataframes.

In [ ]:
# Gather all the record set @ids and load each as dataframe by @id
record_set_ids = [rs.id for rs in metadata.record_sets]

dataframes = {}
for recset_id in record_set_ids:
    records_iter = dataset.records(record_set=recset_id)
    df = pd.DataFrame(list(records_iter))
    dataframes[recset_id] = df
    print(f"Loaded dataframe for record set @id: {recset_id}, shape: {df.shape}")

# Display columns for the first record set
if len(record_set_ids) > 0:
    main_recset_id = record_set_ids[0]
    print(f"Columns in record set {main_recset_id}:\n", dataframes[main_recset_id].columns.tolist())
    display(dataframes[main_recset_id].head())

## 4. Exploratory Data Analysis (EDA)
Demonstrate typical EDA steps: filtering, normalization, and grouping, referencing all columns by their Croissant field or column `@id` as available.

In [ ]:
# Choose dataframe and field @ids for numeric and group operations
# Update these @ids according to previously printed overview

selected_record_set_id = record_set_ids[0]
df = dataframes[selected_record_set_id]

# Try to auto-detect a numeric field
numeric_candidates = []
for col in df.columns:
    try:
        df[col].astype(float)
        numeric_candidates.append(col)
    except Exception:
        pass

if numeric_candidates:
    numeric_field_id = numeric_candidates[0]
    print(f"Using numeric field: {numeric_field_id}")
else:
    print("No numeric field found. Please adjust the field selection.")
    numeric_field_id = df.columns[0]

# Filter based on numeric_field_id
try:
    values = pd.to_numeric(df[numeric_field_id], errors='coerce')
    threshold = values.mean() if not values.isnull().all() else 0
    filtered_df = df[values > threshold].copy()
    print(f"Filtered records where {numeric_field_id} > mean ({threshold:.2f}): {len(filtered_df)} rows")

    filtered_df[f"{numeric_field_id}_normalized"] = (values - values.mean()) / values.std()
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Try to group by a likely categorical/textual field
    group_candidates = [col for col in df.columns if col != numeric_field_id]
    group_field_id = group_candidates[0] if group_candidates else numeric_field_id
    if group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
        print(f"Mean of normalized {numeric_field_id} by {group_field_id}:")
        print(grouped_df.head())
except Exception as e:
    print(f"EDA error: {str(e)}")

## 5. Visualization

Visualize a histogram of the numeric field or a grouped barplot, referencing all fields by their `@id` as identified earlier.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot histogram if numeric
if numeric_candidates:
    values_clean = pd.to_numeric(df[numeric_field_id], errors='coerce').dropna()
    plt.figure(figsize=(8, 4))
    sns.histplot(values_clean, bins=30, kde=True)
    plt.title(f"Distribution of field {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')
    plt.tight_layout()
    plt.show()

# If grouped_df exists, show mean plot
if 'grouped_df' in locals() and not grouped_df.empty:
    plt.figure(figsize=(10, 4))
    grouped_df[numeric_field_id].head(20).plot(kind='bar')
    plt.title(f"Mean {numeric_field_id} by {group_field_id}")
    plt.ylabel(f"Mean {numeric_field_id}")
    plt.xlabel(group_field_id)
    plt.tight_layout()
    plt.show()

## 6. Conclusion

* In this notebook, we've loaded the FAIR² dataset using the Croissant schema and the `mlcroissant` library, explored the available record sets by their `@id`, loaded the data as pandas DataFrames, and performed basic filtering, normalization, and visualization steps referencing all dataset elements by their unique `@id`.
* This approach provides a reproducible standardized workflow for working with Croissant-described datasets and can be adapted to any dataset following the Croissant specification.

**Next steps:**
- Experiment with more advanced EDA and machine learning workflows,
- Explore additional record sets and fields by their `@id`s,
- Consult the Croissant metadata for richer semantic context and documented field meanings.